# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [1]:
%load_ext autoreload

In [2]:
%autoreload 2

import pathlib
import polars as pl
import torch
from torch_geometric.data import HeteroData
from torch_geometric.utils import to_undirected
from torch_geometric.loader import LinkNeighborLoader

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsSeqRecDataModule,
    bipartite_graph_preprocess_dataset,
    fetch_dataset,
    fetch_metadata,
)

/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: dlopen(/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so, 0x0006): Library not loaded: /Library/Frameworks/Python.framework/Versions/3.12/Python
  Referenced from: <441E30E4-F1D4-325A-924A-8C4E5BD0FA29> /Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so
  Reason: tried: '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file)
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/Users/haru256/repo/github.com/haru-256/

In [3]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

2025-06-15 04:45:23.613 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:fetch_dataset:39 - Fetching Amazon Reviews 2023 dataset


user_id,parent_asin,rating,timestamp,history
str,str,str,str,str
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07SRWRH5D""","""5.0""","""1587051114941""",""""""
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""","""B07DK1H3H5""","""4.0""","""1608186804795""","""B07SRWRH5D"""
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""","""B07MFMFW34""","""5.0""","""1490877431000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B00HUWA45W""","""5.0""","""1427591932000""",""""""
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""","""B0BCHWZX95""","""5.0""","""1577637634017""","""B00HUWA45W"""


In [4]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

Dataset Size
train: 3847041, valid: 344592, test: 363867


The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [5]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(["parent_asin", "title", "categories"])
metadata_df.head(5)

2025-06-15 04:45:26.272 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:fetch_metadata:59 - Fetching Amazon Reviews 2023 metadata


parent_asin,title,categories
str,str,list[str]
"""B000FH0MHO""","""Dash 8-300 Professional Add-On""","[""Video Games"", ""PC"", ""Games""]"
"""B00069EVOG""","""Phantasmagoria: A Puzzle of Fl…","[""Video Games"", ""PC"", ""Games""]"
"""B00Z9TLVK0""","""NBA 2K17 - Early Tip Off Editi…","[""Video Games"", ""PlayStation 4"", ""Games""]"
"""B07SZJZV88""","""Nintendo Selects: The Legend o…","[""Video Games"", ""Legacy Systems"", … ""Games""]"
"""B002WH4ZJG""","""Thrustmaster Elite Fitness Pac…","[""Video Games"", ""Legacy Systems"", … ""Fitness Accessories""]"


In [6]:
print(f"Parent Asin Size: {len(metadata_df)}")

Parent Asin Size: 137269


The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [7]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

2025-06-15 04:45:27.796 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:prepare_data:682 - Loading preprocessed dataset


In [8]:
datamodule.train_df.head(5)

user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,history,history_index,history_category,history_category_index
str,i64,str,i64,str,i64,f64,i64,list[str],list[i64],list[str],list[i64]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07SRWRH5D""",30008,"""Video Games/PlayStation 4/Game…",164,5.0,1587051114941,[],[],[],[]
"""AGCI7FAH4GL5FI65HYLKWTMFZ2CQ""",1216183,"""B07DK1H3H5""",27742,"""Video Games/PC/Games""",146,4.0,1608186804795,"[""B07SRWRH5D""]",[30008],"[""Video Games/PlayStation 4/Games""]",[164]
"""AGXVBIUFLFGMVLATYXHJYL4A5Q7Q""",1576453,"""B07MFMFW34""",29081,"""Video Games/PC/Games""",146,5.0,1490877431000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961308,"""B00HUWA45W""",19292,"""Video Games/Xbox One/Accessori…",177,5.0,1427591932000,[],[],[],[]
"""AFTC6ZR5IKNRDG5JCPVNVMU3XV2Q""",961308,"""B0BCHWZX95""",33732,"""Video Games/Nintendo Switch/Ac…",121,5.0,1577637634017,"[""B00HUWA45W""]",[19292],"[""Video Games/Xbox One/Accessories""]",[177]


In [9]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

(tensor([1948178, 1258638]),
 tensor([16973, 30480]),
 tensor([[21180,  5833],
         [17200, 20326]]),
 tensor([[0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0]]))

## Bipartite Graph

In [10]:
(
    df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

2025-06-15 04:45:59.496 | INFO     | ml_sandbox_libs.data.amazon_reviews_dataset:bipartite_graph_preprocess_dataset:882 - Preprocessing the dataset for bipartite graph


In [11]:
df.head(5)

split,user_id,user_index,parent_asin,item_index,category,category_index,rating,timestamp,num_ratings
str,str,i64,str,i64,str,i64,f64,i64,u32
"""train""","""AHL6RJONCBPWPF3R556HXSXPAV7Q""",1900071,"""B09V5R5LSZ""",33507,"""Video Games/Xbox One/Downloada…",192,5.0,1456845425000,1
"""train""","""AEXJ3Q7F3MZAVJHYLCXF2VPECAWQ""",494076,"""B09QL4J4NW""",33415,"""Video Games/PlayStation 4/Acce…",153,5.0,1533601697865,1
"""valid""","""AFDL32HCGFSQ56I3KBA2RG3OMZVQ""",1,"""B0714J4D8X""",1,"""Video Games/Nintendo Switch/Ac…",121,5.0,1647663136758,1
"""train""","""AGNZULNYF2A7UW6K6URVEPZ7CUPA""",1409785,"""B0819CCCN6""",31116,"""Video Games/Nintendo Switch/Ac…",128,5.0,1601700394018,1
"""train""","""AFXRPKGAFQWDFPKHLTPKVBDAJQRA""",1036220,"""B07731LM4G""",26630,"""Video Games/PC/Games""",146,1.0,1533264184219,1


In [12]:
edge_index = torch.tensor(
    df.filter(pl.col("split") == "train").select(["user_index", "item_index"]).to_numpy()
).T.contiguous()
val_edge_label_index = torch.tensor(
    df.filter(pl.col("split") == "valid").select(["user_index", "item_index"]).to_numpy()
).T.contiguous()
test_edge_label_index = torch.tensor(
    df.filter(pl.col("split") == "test").select(["user_index", "item_index"]).to_numpy()
).T.contiguous()


In [13]:
data = HeteroData()
data["user"].num_nodes = len(user2index)
data["item"].num_nodes = len(item2index)
# メッセージパッシングに使う学習用エッジを無向グラフに変換
data["user", "rates", "item"].edge_index = to_undirected(edge_index)
# ラベルは評価に使うので無向グラフに変換しない
data["user", "rates", "item"].val_edge_label_index = val_edge_label_index
data["user", "rates", "item"].test_edge_label_index = test_edge_label_index


In [14]:
# 学習用ローダー
# 'edge_label_index'に対象となるエッジを指定し、そこからサンプリングを開始する
train_loader = LinkNeighborLoader(
    data=data,
    num_neighbors={('user', 'rates', 'item'): [10, 5]}, # 1ホップ先で10個、2ホップ先で5個のneighborをサンプリング
    edge_label_index=('user', 'rates', 'item'), # 学習用のエッジを指定
    batch_size=128,
    shuffle=True,
    # ネガティブサンプリングも自動で行う
    neg_sampling_ratio=1.0 # ポジティブエッジ1つにつきネガティブエッジを1つサンプリング
)

# # 検証用ローダー（ネガティブサンプリングやシャッフルは不要）
# val_loader = LinkNeighborLoader(
#     data=data,
#     num_neighbors={('user'): [10, 5], ('item'): [10, 5]},
#     edge_label_index=data['user', 'rates', 'item'].val_edge_label_index, # 検証用のエッジを指定
#     batch_size=128,
#     shuffle=False,
#     neg_sampling_ratio=0.0 # 評価時はポジティブエッジのみ
# )
 

In [16]:
batch = next(iter(train_loader))

ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [15]:
import pyg_lib

OSError: dlopen(/Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so, 0x0006): Library not loaded: /Library/Frameworks/Python.framework/Versions/3.12/Python
  Referenced from: <441E30E4-F1D4-325A-924A-8C4E5BD0FA29> /Users/haru256/repo/github.com/haru-256/ml-sandbox/libs/ml_sandbox_libs/.venv/lib/python3.12/site-packages/libpyg.so
  Reason: tried: '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file), '/Library/Frameworks/Python.framework/Versions/3.12/Python' (no such file)